In [19]:
import xarray as xr
import pandas as pd

In [20]:
# Open the NetCDF file
file_path = "/Users/Mbongeleni/Library/CloudStorage/OneDrive-StellenboschUniversity/Documents/CSIR/Mercator/CSIR_ML6_Merged.nc"
ds = xr.open_dataset(file_path)

In [21]:
# Inspect the dataset structure
print(ds)

# Convert to a DataFrame to see columns
df = ds.to_dataframe().reset_index()

# Show the first 10 rows and the column names
print(df.head(100))
print(df.columns)


<xarray.Dataset> Size: 2MB
Dimensions:         (time: 133, latitude: 35, longitude: 10)
Coordinates:
  * time            (time) datetime64[ns] 1kB 2009-12-01 ... 2020-12-01
  * latitude        (latitude) float64 280B -69.5 -68.5 -67.5 ... -36.5 -35.5
  * longitude       (longitude) float64 80B 15.5 16.5 17.5 ... 22.5 23.5 24.5
Data variables:
    fgco2           (time, latitude, longitude) float64 372kB ...
    pCO2            (time, latitude, longitude) float64 372kB ...
    depth           (time, latitude, longitude) float32 186kB ...
    latitude_merc   (time, latitude, longitude) float32 186kB ...
    longitude_merc  (time, latitude, longitude) float32 186kB ...
    so              (time, latitude, longitude) float32 186kB ...
    thetao          (time, latitude, longitude) float32 186kB ...
    AT              (time, latitude, longitude) float32 186kB ...
         time  latitude  longitude         fgco2        pCO2     depth  \
0  2009-12-01     -69.5       15.5  1.257129e-09  379

In [22]:
# Drop unwanted columns
df = df.drop(columns=["latitude_merc", "longitude_merc"])

# Show updated columns
print("Updated columns:", df.columns.tolist())

# Preview first rows
print(df.head())

Updated columns: ['time', 'latitude', 'longitude', 'fgco2', 'pCO2', 'depth', 'so', 'thetao', 'AT']
        time  latitude  longitude         fgco2        pCO2     depth  \
0 2009-12-01     -69.5       15.5  1.257129e-09  379.237760  3.819495   
1 2009-12-01     -69.5       16.5  1.625722e-09  379.463568  3.819495   
2 2009-12-01     -69.5       17.5  1.083595e-09  374.479944  3.819495   
3 2009-12-01     -69.5       18.5 -1.756832e-09  367.623003  3.819495   
4 2009-12-01     -69.5       19.5 -1.511893e-09  367.727819  3.819495   

          so    thetao           AT  
0  32.952362 -1.796594  2261.027832  
1  33.324688 -1.800256  2276.632324  
2  33.153782 -1.661092  2268.790283  
3  33.367413 -1.459670  2277.037598  
4  33.584095 -1.759239  2287.787109  


In [23]:
# Show original column names
print("Original columns:", df.columns.tolist())

# Define new column names
new_columns = [
    "Date", "Lat", "longitude", "fgCO2", "pCO2",
    "DEPTH(M)", "Salinity",
    "Temperature", "Total Alkalinity"
]

# Rename columns (make sure lengths match)
df.columns = new_columns[:len(df.columns)]

# Show updated column names
print("Renamed columns:", df.columns.tolist())

# Preview the first rows
print(df.head())

Original columns: ['time', 'latitude', 'longitude', 'fgco2', 'pCO2', 'depth', 'so', 'thetao', 'AT']
Renamed columns: ['Date', 'Lat', 'longitude', 'fgCO2', 'pCO2', 'DEPTH(M)', 'Salinity', 'Temperature', 'Total Alkalinity']
        Date   Lat  longitude         fgCO2        pCO2  DEPTH(M)   Salinity  \
0 2009-12-01 -69.5       15.5  1.257129e-09  379.237760  3.819495  32.952362   
1 2009-12-01 -69.5       16.5  1.625722e-09  379.463568  3.819495  33.324688   
2 2009-12-01 -69.5       17.5  1.083595e-09  374.479944  3.819495  33.153782   
3 2009-12-01 -69.5       18.5 -1.756832e-09  367.623003  3.819495  33.367413   
4 2009-12-01 -69.5       19.5 -1.511893e-09  367.727819  3.819495  33.584095   

   Temperature  Total Alkalinity  
0    -1.796594       2261.027832  
1    -1.800256       2276.632324  
2    -1.661092       2268.790283  
3    -1.459670       2277.037598  
4    -1.759239       2287.787109  


In [24]:
# Ensure Date column is datetime
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# Create new columns
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day

# Preview the result
print(df[["Date", "Year", "Month", "Day"]].head(10))

        Date  Year  Month  Day
0 2009-12-01  2009     12    1
1 2009-12-01  2009     12    1
2 2009-12-01  2009     12    1
3 2009-12-01  2009     12    1
4 2009-12-01  2009     12    1
5 2009-12-01  2009     12    1
6 2009-12-01  2009     12    1
7 2009-12-01  2009     12    1
8 2009-12-01  2009     12    1
9 2009-12-01  2009     12    1


In [25]:
# Show updated column names
print("Renamed columns:", df.columns.tolist())

Renamed columns: ['Date', 'Lat', 'longitude', 'fgCO2', 'pCO2', 'DEPTH(M)', 'Salinity', 'Temperature', 'Total Alkalinity', 'Year', 'Month', 'Day']


In [26]:
# Define function to classify season
def classify_season(month):
    if month in [12, 1, 2]:
        return "Summer"
    elif month in [3, 4, 5]:
        return "Autumn"
    elif month in [6, 7, 8]:
        return "Winter"
    elif month in [9, 10, 11]:
        return "Spring"
    else:
        return None

# Apply classification
df["Season"] = df["Month"].apply(classify_season)

# Preview the result
print(df[["Date", "Year", "Month", "Day", "Season"]].head(12))

         Date  Year  Month  Day  Season
0  2009-12-01  2009     12    1  Summer
1  2009-12-01  2009     12    1  Summer
2  2009-12-01  2009     12    1  Summer
3  2009-12-01  2009     12    1  Summer
4  2009-12-01  2009     12    1  Summer
5  2009-12-01  2009     12    1  Summer
6  2009-12-01  2009     12    1  Summer
7  2009-12-01  2009     12    1  Summer
8  2009-12-01  2009     12    1  Summer
9  2009-12-01  2009     12    1  Summer
10 2009-12-01  2009     12    1  Summer
11 2009-12-01  2009     12    1  Summer
